# Public PPE Baseline

This external pretrained PPE model is used for qualitative comparison and quick demo inference. Its class map may differ from the project MVP schema.

This notebook does **not** train the public model by default. It is for qualitative inference and runtime notes.

## Environment Setup

In [ ]:
from pathlib import Path
import json
import os
import random
import sys
import time


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "dataset").exists() and (candidate / "src").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError(
        "Could not locate project root. Run this notebook from factory-safety-ai-cctv/ "
        "or open it from inside that project."
    )


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

try:
    from ultralytics import YOLO
except ImportError as exc:
    YOLO = None
    print("ERROR: Ultralytics is required for model experiments.")
    print("Install with: pip install ultralytics")

try:
    import torch
    DEVICE = 0 if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print(f"Device: {DEVICE}")

## Dataset Config Check

Dataset YAML is recorded for context, but mAP is not computed unless label spaces are aligned.

In [ ]:
DATA_YAML_CANDIDATES = [
    Path("data/processed/ppe_yolo/data.yaml"),
    Path("dataset/roboflow_hard_hat_workers_yolov11_raw/data.yaml"),
]
IMAGE_EXTENSIONS_FOR_DATA_CHECK = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
PROJECT_CLASS_NAMES = ["person", "helmet", "head"]


def _read_simple_yaml_field(yaml_path: Path, key: str) -> str | None:
    if not yaml_path.exists():
        return None
    for raw_line in yaml_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or ":" not in line:
            continue
        current_key, value = line.split(":", 1)
        if current_key.strip() == key:
            return value.strip().strip("'\"")
    return None


def _count_images(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(1 for item in path.rglob("*") if item.suffix.lower() in IMAGE_EXTENSIONS_FOR_DATA_CHECK)


def _candidate_image_dir(yaml_path: Path, split_key: str) -> Path | None:
    split_value = _read_simple_yaml_field(yaml_path, split_key)
    if not split_value:
        return None
    path_value = _read_simple_yaml_field(yaml_path, "path")
    roots = []
    if path_value:
        roots.append(Path(path_value))
        roots.append(yaml_path.parent / path_value)
    roots.append(yaml_path.parent)
    for root in roots:
        candidate = (root / split_value).resolve()
        if candidate.exists():
            return candidate
    return (yaml_path.parent / split_value).resolve()


def _extract_names_text(yaml_path: Path) -> str:
    text = yaml_path.read_text(encoding="utf-8", errors="ignore")
    for line in text.splitlines():
        if line.strip().startswith("names:"):
            return line.strip()
    return "names: TBD"


def _warn_if_label_map_differs(yaml_path: Path) -> None:
    text = yaml_path.read_text(encoding="utf-8", errors="ignore").lower()
    compact = text.replace('"', "'").replace(" ", "")
    if "['head','helmet','person']" in compact:
        print("WARNING: Roboflow fallback class order appears to be ['head', 'helmet', 'person'].")
        print("Project final schema is ['person', 'helmet', 'head']. Use fallback for smoke experiments only until labels are aligned.")


def _write_runtime_roboflow_yaml(raw_yaml: Path) -> Path | None:
    raw_root = raw_yaml.parent
    train_dir = raw_root / "train" / "images"
    val_dir = raw_root / "valid" / "images"
    if not val_dir.exists():
        val_dir = raw_root / "val" / "images"
    if not val_dir.exists():
        val_dir = raw_root / "test" / "images"
    test_dir = raw_root / "test" / "images"
    if _count_images(train_dir) == 0 or _count_images(val_dir) == 0:
        return None

    runtime_dir = Path("experiments/runtime_dataset_yaml")
    runtime_dir.mkdir(parents=True, exist_ok=True)
    runtime_yaml = runtime_dir / "roboflow_hard_hat_workers_yolov11_runtime.yaml"
    names_text = _extract_names_text(raw_yaml)
    runtime_yaml.write_text(
        "\n".join(
            [
                "path: dataset/roboflow_hard_hat_workers_yolov11_raw",
                "train: train/images",
                f"val: {val_dir.relative_to(raw_root).as_posix()}",
                f"test: {test_dir.relative_to(raw_root).as_posix() if test_dir.exists() else val_dir.relative_to(raw_root).as_posix()}",
                "",
                "nc: 3",
                names_text,
                "",
            ]
        ),
        encoding="utf-8",
    )
    print(f"Created runtime Roboflow YAML: {runtime_yaml}")
    return runtime_yaml


def _usable_yaml(yaml_path: Path) -> Path | None:
    train_dir = _candidate_image_dir(yaml_path, "train")
    val_dir = _candidate_image_dir(yaml_path, "val")
    train_count = _count_images(train_dir) if train_dir else 0
    val_count = _count_images(val_dir) if val_dir else 0
    print(f"Checking {yaml_path}: train_images={train_count}, val_images={val_count}")
    if train_count > 0 and val_count > 0:
        return yaml_path
    if yaml_path.as_posix().endswith("dataset/roboflow_hard_hat_workers_yolov11_raw/data.yaml"):
        return _write_runtime_roboflow_yaml(yaml_path)
    return None


def select_data_yaml() -> Path | None:
    for candidate in DATA_YAML_CANDIDATES:
        if not candidate.exists():
            print(f"Dataset YAML candidate not found: {candidate}")
            continue
        usable = _usable_yaml(candidate)
        if usable is not None:
            return usable
        print(f"Skipping unusable dataset YAML: {candidate}")
    return None


DATA_YAML = select_data_yaml()
if DATA_YAML is None:
    print("ERROR: No usable dataset YAML found.")
    print("Expected a YAML with non-empty train/val image folders from one of:")
    for candidate in DATA_YAML_CANDIDATES:
        print(f"  - {candidate}")
else:
    print(f"Selected DATA_YAML: {DATA_YAML}")
    print(DATA_YAML.read_text(encoding="utf-8"))
    _warn_if_label_map_differs(DATA_YAML)

## Load Public PPE Model

`PUBLIC_MODEL = "hf://Hexmon/vyra-yolo-ppe-detection/best.pt"`

In [ ]:
MODEL_ID = "public_ppe_baseline"
PUBLIC_MODEL = "hf://Hexmon/vyra-yolo-ppe-detection/best.pt"
BASE_MODEL = PUBLIC_MODEL
ROLE = "external_baseline"
EXPERIMENT_DIR = Path("experiments/public_ppe_baseline")
RUNS_DIR = EXPERIMENT_DIR / "runs"
METRICS_PATH = EXPERIMENT_DIR / "metrics.json"

model = None
model_load_error = None
runtime_metrics = {
    "avg_inference_ms_per_image": "TBD",
    "fps_estimate": "TBD",
}
video_output_dirs = {}
if YOLO is None:
    model_load_error = "Ultralytics is not installed."
    print("Skipping model load because Ultralytics is unavailable.")
else:
    try:
        model = YOLO(PUBLIC_MODEL)
        print("Public model loaded:", PUBLIC_MODEL)
        print("Model names:", getattr(model, "names", "N/A"))
    except Exception as exc:
        model_load_error = str(exc)
        print(f"Public model load failed gracefully: {exc}")

## Public Class Mapping Warning

The public PPE model has richer PPE classes than the project MVP schema. For MVP, only Person, Hardhat, and NO-Hardhat are mapped into the risk signal schema. Other classes are ignored unless future rules use them.

In [ ]:
PUBLIC_TO_PROJECT_CLASS = {
    "Person": "person",
    "Hardhat": "helmet",
    "NO-Hardhat": "head",
}
class_names = getattr(model, "names", {}) if model is not None else {}
if isinstance(class_names, dict):
    public_names = list(class_names.values())
else:
    public_names = list(class_names)

mapped_classes = [name for name in public_names if name in PUBLIC_TO_PROJECT_CLASS]
ignored_classes = [name for name in public_names if name not in PUBLIC_TO_PROJECT_CLASS]

print("Mapped classes:", mapped_classes)
print("Ignored classes:", ignored_classes)

## Sample Prediction and Inference Speed

Use the same shared sample images as the project-trained candidates.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SHARED_SAMPLE_PATH = Path("experiments/shared_eval_samples.json")
RANDOM_SEED = 42
SAMPLE_COUNT = 12


def _collect_images(root: Path) -> list[Path]:
    if not root.exists():
        return []
    return sorted(path for path in root.rglob("*") if path.suffix.lower() in IMAGE_EXTENSIONS)


def get_shared_sample_images(sample_count: int = SAMPLE_COUNT, seed: int = RANDOM_SEED) -> list[Path]:
    if SHARED_SAMPLE_PATH.exists():
        data = json.loads(SHARED_SAMPLE_PATH.read_text(encoding="utf-8"))
        images = [Path(item) for item in data.get("sample_images", [])]
        existing = [path for path in images if path.exists()]
        if existing:
            print(f"Reusing shared sample list: {SHARED_SAMPLE_PATH}")
            return existing[:sample_count]
        print("Shared sample file exists but no listed images were found. Rebuilding it.")

    candidates = []
    search_roots = [
        Path("data/processed/ppe_yolo/images/test"),
        Path("dataset/roboflow_hard_hat_workers_yolov11_raw/test/images"),
        Path("dataset"),
    ]
    for root in search_roots:
        candidates = _collect_images(root)
        if candidates:
            print(f"Sample source: {root}")
            break

    if not candidates:
        print("WARNING: No sample images found.")
        SHARED_SAMPLE_PATH.parent.mkdir(parents=True, exist_ok=True)
        SHARED_SAMPLE_PATH.write_text(json.dumps({"sample_images": []}, indent=2), encoding="utf-8")
        return []

    rng = random.Random(seed)
    rng.shuffle(candidates)
    selected = sorted(candidates[:sample_count])
    SHARED_SAMPLE_PATH.parent.mkdir(parents=True, exist_ok=True)
    SHARED_SAMPLE_PATH.write_text(
        json.dumps({"seed": seed, "sample_images": [str(path) for path in selected]}, indent=2),
        encoding="utf-8",
    )
    print(f"Saved shared sample list: {SHARED_SAMPLE_PATH}")
    return selected


sample_images = get_shared_sample_images()
print(f"Sample count: {len(sample_images)}")
for path in sample_images[:5]:
    print(" -", path)

In [ ]:
def metric_value(obj, attr_path: str, default="TBD"):
    current = obj
    for attr in attr_path.split("."):
        if current is None:
            return default
        current = getattr(current, attr, None)
    if current is None:
        return default
    try:
        if callable(current):
            current = current()
    except TypeError:
        pass
    try:
        return float(current)
    except (TypeError, ValueError):
        return current


def load_metrics(path: Path, fallback: dict) -> dict:
    if path.exists():
        try:
            existing = json.loads(path.read_text(encoding="utf-8"))
            merged = fallback.copy()
            merged.update(existing)
            return merged
        except json.JSONDecodeError:
            print(f"WARNING: Could not parse existing metrics file: {path}")
    return fallback


def save_metrics(metrics: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    print(f"Saved metrics: {path}")


runtime_metrics = {
    "avg_inference_ms_per_image": "TBD",
    "fps_estimate": "TBD",
}

if model is None:
    print("Skipping public model prediction because model is not loaded.")
elif not sample_images:
    print("No sample images available; skipping public model prediction.")
else:
    start = time.perf_counter()
    results = model.predict(
        source=[str(path) for path in sample_images],
        conf=0.4,
        save=True,
        project="experiments/public_ppe_baseline/runs",
        name="predict_samples",
        exist_ok=True,
    )
    elapsed = time.perf_counter() - start
    avg_ms = (elapsed / len(sample_images)) * 1000
    fps = 1000 / avg_ms if avg_ms > 0 else None
    runtime_metrics = {
        "avg_inference_ms_per_image": avg_ms,
        "fps_estimate": fps,
    }
    print(f"Saved predictions to: experiments/public_ppe_baseline/runs/predict_samples")
    print(f"Average inference: {avg_ms:.2f} ms/image")
    print(f"FPS estimate: {fps:.2f}" if fps else "FPS estimate: N/A")

## Optional Video Prediction

In [ ]:
VIDEO_SOURCES = {
    "camera_1": Path("video_source/camera_1/cam1.mp4"),
    "camera_2": Path("video_source/camera_2/cam2.mp4"),
    "camera_3": Path("video_source/camera_3/cam3.mp4"),
}

video_output_dirs = {}
if model is None:
    print("Skipping video prediction because model is not loaded.")
for camera_id, video_path in ([] if model is None else VIDEO_SOURCES.items()):
    if not video_path.exists():
        print(f"Skipping {camera_id}; video not found: {video_path}")
        continue
    output_name = f"predict_video_{camera_id}"
    model.predict(
        source=str(video_path),
        conf=0.4,
        save=True,
        project="experiments/public_ppe_baseline/runs",
        name=output_name,
        exist_ok=True,
    )
    video_output_dirs[camera_id] = f"experiments/public_ppe_baseline/runs/{output_name}"

## Save Metadata / Pseudo Metrics

No mAP is reported because the public model label space may not align with the project dataset.

In [ ]:
def metric_value(obj, attr_path: str, default="TBD"):
    current = obj
    for attr in attr_path.split("."):
        if current is None:
            return default
        current = getattr(current, attr, None)
    if current is None:
        return default
    try:
        if callable(current):
            current = current()
    except TypeError:
        pass
    try:
        return float(current)
    except (TypeError, ValueError):
        return current


def load_metrics(path: Path, fallback: dict) -> dict:
    if path.exists():
        try:
            existing = json.loads(path.read_text(encoding="utf-8"))
            merged = fallback.copy()
            merged.update(existing)
            return merged
        except json.JSONDecodeError:
            print(f"WARNING: Could not parse existing metrics file: {path}")
    return fallback


def save_metrics(metrics: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    print(f"Saved metrics: {path}")


metrics_payload = {
    "model_id": MODEL_ID,
    "base_model": BASE_MODEL,
    "role": ROLE,
    "dataset_yaml": str(DATA_YAML) if DATA_YAML else "TBD",
    "status": "qualitative_inference_completed" if (model is not None and sample_images) else ("model_load_failed" if model_load_error else "not_run"),
    "errors": {
        "model_load_error": model_load_error,
    },
    "class_names": class_names,
    "class_mapping": PUBLIC_TO_PROJECT_CLASS,
    "mapped_classes": mapped_classes,
    "ignored_classes": ignored_classes,
    "notes": "External pretrained PPE baseline. Label space differs from project MVP schema.",
    "qualitative_only": True,
    "train_config": {
        "epochs": "N/A",
        "imgsz": "N/A",
        "batch": "N/A",
        "device": "N/A",
    },
    "dataset_metrics": {
        "mAP50": "N/A",
        "mAP50_95": "N/A",
        "precision": "N/A",
        "recall": "N/A",
    },
    "runtime_metrics": runtime_metrics,
    "qualitative_outputs": {
        "sample_prediction_dir": "experiments/public_ppe_baseline/runs/predict_samples",
        "video_prediction_dir": video_output_dirs if video_output_dirs else "TBD",
    },
    "risk_signal_readiness": {
        "person_detection_supported": "Person" in mapped_classes,
        "helmet_detection_supported": "Hardhat" in mapped_classes,
        "no_helmet_or_head_supported": "NO-Hardhat" in mapped_classes,
        "danger_zone_supported_by_postprocessing": True,
        "notes": "Person -> person_detected; Hardhat -> helmet_detected; NO-Hardhat -> head_detected/no_helmet_confidence.",
    },
    "selection_notes": "Use as qualitative demo fallback or external reference only until licensing, class mapping, and evaluation are aligned.",
}
save_metrics(metrics_payload, METRICS_PATH)

## Risk Signal Bridge Note

- `Person` -> `person_detected`
- `Hardhat` -> `helmet_detected`
- `NO-Hardhat` -> `head_detected` / `no_helmet_confidence`

This notebook does not implement final Risk Scoring.